# Dividing POIs into sub-groups

In [5]:
import pandas as pd

# 1. Load the existing unique POI file
pois = pd.read_csv("../data/processed/pois_unique.csv")

# 2. Define which poi_type belongs to which group
POI_GROUP_MAPPING = {
    "bus_stop":      "Accessibility",
    "parking_space": "Accessibility",

    "cinema":     "Anchor_Destinations",
    "museum":     "Anchor_Destinations",
    "temple":     "Anchor_Destinations",
    "recreation": "Anchor_Destinations",

    "office":  "Daytime_Population",
    "college": "Daytime_Population",
    "school":  "Daytime_Population",
    "hospital":  "Daytime_Population",
    "clinic":  "Daytime_Population",

    "retail": "Commercial_Vitality",
    "bank":   "Commercial_Vitality"

}

# 3. Add the new column
pois["poi_group"] = pois["poi_type"].map(POI_GROUP_MAPPING)

# 4. Sanity check — make sure every poi_type got mapped (no blanks)
unmapped = pois[pois["poi_group"].isna()]["poi_type"].unique()
print("Unmapped poi_types:", unmapped)  # should print an empty array

# 5. Save as a new file
pois.to_csv("../data/processed/pois_grouped.csv", index=False)
print("Saved:", pois.shape)


Unmapped poi_types: []
Saved: (13809, 9)


In [6]:
import pandas as pd

df=pd.read_csv("../data/processed/pois_grouped.csv")
df

,place_id,poi_type,name,primary_type,latitude,longitude,areas_found_in,n_areas,poi_group
0,ChIJl2BEeZcZ6zkRMtRMoN7ZwUs,bank,Nepal Cooperative Financial Institution,bank,27.693853,85.337153,Baneshwor,1,Commercial_Vitality
1,ChIJY0HOeJcZ6zkRawb6tJIu3f4,bank,Vastabik SACCOS,bank,27.693881,85.336996,Baneshwor,1,Commercial_Vitality
2,ChIJVVb51pkZ6zkRMtFPxqztdAY,bank,Global IME Bank Limited,bank,27.694435,85.337392,Baneshwor,1,Commercial_Vitality
3,ChIJdyrqzZkZ6zkRYRaFBexay1A,bank,Public Sutradhar Multipurpose Co-op.,bank,27.695078,85.337137,Baneshwor,1,Commercial_Vitality
4,ChIJXyQczZkZ6zkR0pOLzWycmhQ,bank,Sutradhar SACCOS,bank,27.695124,85.336964,Baneshwor,1,Commercial_Vitality
...,...,...,...,...,...,...,...,...,...
13804,ChIJQzQKKQAZ6zkRtGlF0_-objs,temple,"Kumari Mandir,Thapathali",hindu_temple,27.691167,85.318148,Pulchowk,1,Anchor_Destinations
13805,ChIJPzw3fzsZ6zkR1L9L7UlECbc,temple,Sansari Mandir,hindu_temple,27.670158,85.307073,Pulchowk,1,Anchor_Destinations
13806,ChIJFVPR7DEY6zkRu-UETNnwY7w,temple,Bhanimandal mahadev temple,hindu_temple,27.672252,85.305086,Pulchowk,1,Anchor_Destinations
13807,ChIJDZWvbgAZ6zkRN7pYWwh6A0Q,temple,Kalika Mandir,hindu_temple,27.685005,85.304379,Pulchowk,1,Anchor_Destinations


# Sub-groups for demand: anchor_destination, daytime_population, commercial_viability 

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/final_dataset.csv")

# ---- the three Demand sub-groups ----
anchor     = ["cinema_count_500m", "museum_count_500m", "temple_count_500m", "recreation_count_500m"]
daytime    = ["office_count_500m", "college_count_500m", "school_count_500m", "hospital_count_500m", "clinic_count_500m"]
commercial = ["retail_count_500m", "bank_count_500m"]

def minmax(col):
    mn, mx = df[col].min(), df[col].max()
    return pd.Series(0.0, index=df.index) if mx == mn else (df[col] - mn) / (mx - mn)

def group_score(features):
    norm = pd.DataFrame({c: minmax(c) for c in features})
    return norm.mean(axis=1).round(4)

# ---- clean feature table: identity + sub-scores, NO raw counts ----
feat = df[["place_id", "restaurant_name", "latitude", "longitude", "search_area"]].copy()
feat["Anchor_Destinations"] = group_score(anchor)
feat["Daytime_Population"]  = group_score(daytime)
feat["Commercial_Vitality"] = group_score(commercial)

feat.to_csv("../data/processed/demand_subgroups.csv", index=False)
feat.head()

,place_id,restaurant_name,latitude,longitude,search_area,Anchor_Destinations,Daytime_Population,Commercial_Vitality
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,JAR - Just Another Restaurant,27.699546,85.337687,Baneshwor,0.1121,0.4828,0.3035
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,27.692262,85.336472,Baneshwor,0.1265,0.5931,0.4827
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Pink Putali Restaurant & Bar,27.689064,85.334295,Baneshwor,0.1779,0.6513,0.5083
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Munch N More Restaurant and Bar,27.681549,85.341340,Baneshwor,0.1264,0.2961,0.2731
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27 Degree North Restaurant,27.688812,85.334080,Baneshwor,0.1803,0.6303,0.4946


# Demand final 

In [10]:
import pandas as pd

d = pd.read_csv("../data/processed/sub_groups/demand_subgroups.csv")

sub = ["Anchor_Destinations", "Daytime_Population", "Commercial_Vitality"]

# nested average -> each sub-group counts equally
d["Demand"] = d[sub].mean(axis=1).round(4)

d.to_csv("../data/processed/sub_groups/demand_features.csv", index=False)
d[["place_id", "search_area"] + sub + ["Demand"]].head()

,place_id,search_area,Anchor_Destinations,Daytime_Population,Commercial_Vitality,Demand
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,Baneshwor,0.1121,0.4828,0.3035,0.2995
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Baneshwor,0.1265,0.5931,0.4827,0.4008
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Baneshwor,0.1779,0.6513,0.5083,0.4458
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Baneshwor,0.1264,0.2961,0.2731,0.2319
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,Baneshwor,0.1803,0.6303,0.4946,0.4351


# For Accessibility

## finding intersections per area

In [2]:
import pandas as pd, numpy as np
from sklearn.neighbors import BallTree

df    = pd.read_csv("../data/processed/final_dataset.csv")
inter = pd.read_csv("../data/raw_data/roads/intersections_all_areas.csv")

# --- clean: drop duplicate junctions at identical coordinates ---
inter = inter.drop_duplicates(subset=["latitude", "longitude"]).reset_index(drop=True)

# --- count intersections within 500m (haversine BallTree, same as your POI counts) ---
EARTH_R = 6_371_000.0            # metres
radius  = 500 / EARTH_R         # radius in radians

tree   = BallTree(np.radians(inter[["latitude", "longitude"]].values), metric="haversine")
counts = tree.query_radius(np.radians(df[["latitude", "longitude"]].values),
                           r=radius, count_only=True)

df["intersection_count_500m"] = counts
df.to_csv("../data/processed/final_dataset.csv", index=False)
df[["place_id", "search_area", "intersection_count_500m"]].head()

,place_id,search_area,intersection_count_500m
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,Baneshwor,138
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Baneshwor,128
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Baneshwor,129
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Baneshwor,364
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,Baneshwor,136


## separating bus_stop, parking and intersection count

In [4]:
import pandas as pd

df = pd.read_csv("../data/processed/final_dataset.csv")

id_cols     = ["place_id", "restaurant_name", "latitude", "longitude", "search_area"]
access_cols = ["bus_stop_count_500m", "parking_space_count_500m", "intersection_count_500m"]

accessibility = df[id_cols + access_cols].copy()
accessibility.to_csv("../data/processed/sub_groups/accessibility_features.csv", index=False)
accessibility.head()

,place_id,restaurant_name,latitude,longitude,search_area,bus_stop_count_500m,parking_space_count_500m,intersection_count_500m
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,JAR - Just Another Restaurant,27.699546,85.337687,Baneshwor,1,5,138
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,27.692262,85.336472,Baneshwor,4,6,128
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Pink Putali Restaurant & Bar,27.689064,85.334295,Baneshwor,6,9,129
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Munch N More Restaurant and Bar,27.681549,85.341340,Baneshwor,0,4,364
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27 Degree North Restaurant,27.688812,85.334080,Baneshwor,6,10,136


### For roads

These provide complementary information:

- Transit access → bus stops
- Parking availability → parking
- Street connectivity → intersections
- Commercial visibility → main road length

In [6]:
import pandas as pd
from shapely import wkt
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from pyproj import Transformer

acc   = pd.read_csv("../data/processed/sub_groups/accessibility_features.csv")
roads = pd.read_csv("../data/raw_data/roads/roads_all_areas.csv")

# 1. keep only MAIN roads: primary / trunk / secondary  (tertiary excluded)
main = roads[roads["highway"].isin(["primary", "trunk", "secondary"])].copy()

# 2. project WGS84 -> UTM 45N (metres) and build line geometries
to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)

def project_line(wkt_str):
    line = wkt.loads(wkt_str)
    xs, ys = to_utm.transform(*line.xy)
    return LineString(zip(xs, ys))

main_lines = [project_line(w) for w in main["geometry_wkt"]]
tree = STRtree(main_lines)

# 3. distance from each location to the NEAREST main-road line (metres)
loc_x, loc_y = to_utm.transform(acc["longitude"].values, acc["latitude"].values)
acc["dist_to_main_road_m"] = [
    round(Point(x, y).distance(main_lines[tree.nearest(Point(x, y))]), 1)
    for x, y in zip(loc_x, loc_y)
]

acc.to_csv("../data/processed/sub_groups/accessibility_features.csv", index=False)
acc.head()

,place_id,restaurant_name,latitude,longitude,search_area,bus_stop_count_500m,parking_space_count_500m,intersection_count_500m,dist_to_main_road_m
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,JAR - Just Another Restaurant,27.699546,85.337687,Baneshwor,1,5,138,71.4
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,27.692262,85.336472,Baneshwor,4,6,128,27.6
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Pink Putali Restaurant & Bar,27.689064,85.334295,Baneshwor,6,9,129,53.0
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Munch N More Restaurant and Bar,27.681549,85.341340,Baneshwor,0,4,364,14.7
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27 Degree North Restaurant,27.688812,85.334080,Baneshwor,6,10,136,20.9


# Accessibility

In [11]:
import pandas as pd

a = pd.read_csv("../data/processed/sub_groups/accessibility_features.csv")

benefit = ["bus_stop_count_500m", "parking_space_count_500m", "intersection_count_500m"]  # more = better
cost    = ["dist_to_main_road_m"]                                                          # less = better

def minmax_benefit(col):
    mn, mx = a[col].min(), a[col].max()
    return pd.Series(0.0, index=a.index) if mx == mn else (a[col] - mn) / (mx - mn)

def minmax_cost(col):        # INVERT: near a main road = high score
    mn, mx = a[col].min(), a[col].max()
    return pd.Series(1.0, index=a.index) if mx == mn else (mx - a[col]) / (mx - mn)

norm = pd.DataFrame(index=a.index)
for c in benefit: norm[c] = minmax_benefit(c)
for c in cost:    norm[c] = minmax_cost(c)

a["Accessibility"] = norm.mean(axis=1).round(4)

a[["place_id","restaurant_name","latitude","longitude","search_area","Accessibility"]] \
    .to_csv("../data/processed/sub_groups/accessibility_factor.csv", index=False)

a.groupby("search_area")["Accessibility"].mean().round(3).sort_values(ascending=False)

search_area
Durbar Marg                0.533
New Road                   0.510
Baneshwor                  0.457
Boudha stupa               0.428
Pulchowk                   0.406
Patan durbar square        0.395
Koteshwor                  0.383
Bhaktapur durbar square    0.355
Kirtipur                   0.272
Name: Accessibility, dtype: float64

# for house density

In [7]:
import pandas as pd, glob, os

files = sorted(glob.glob("../data/raw_data/house_density/buildings_*.csv"))

frames = []
for f in files:
    d = pd.read_csv(f)
    area = os.path.basename(f).replace("buildings_", "").replace(".csv", "")   # area from filename
    d["area"] = area
    frames.append(d)

buildings = pd.concat(frames, ignore_index=True)

# cleaning = keep only what we use (drop building_type + name), dedup coords
buildings = buildings[["area", "latitude", "longitude", "building_area_sqm"]]
buildings = buildings.drop_duplicates(subset=["latitude", "longitude"]).reset_index(drop=True)

buildings.to_csv("../data/processed/buildings_combined.csv", index=False)
print("combined:", buildings.shape)
buildings["area"].value_counts()

combined: (143625, 4)


area
baneshwor                  24953
durbarmarg                 24220
bouddha                    17935
new_road                   17432
patan                      15658
bhaktapur durbar square    14891
kirtipur                   11495
koteshwor                  10892
pulchowk                    6149
Name: count, dtype: int64

In [9]:
import pandas as pd, numpy as np
from sklearn.neighbors import BallTree

acc       = pd.read_csv("../data/processed/final_dataset.csv")
buildings = pd.read_csv("../data/processed/buildings_combined.csv")

EARTH_R = 6_371_000.0
radius  = 500 / EARTH_R

tree   = BallTree(np.radians(buildings[["latitude", "longitude"]].values), metric="haversine")
counts = tree.query_radius(np.radians(acc[["latitude", "longitude"]].values),
                           r=radius, count_only=True)

acc["building_count_500m"] = counts

# ---- keep only identity + building count; drop all POI/other counts ----
id_cols = ["place_id", "restaurant_name", "latitude", "longitude", "search_area"]
house_density = acc[id_cols + ["building_count_500m"]].copy()

house_density.to_csv("../data/processed/sub_groups/house_density_features.csv", index=False)

# sanity check — EVERY area should be non-zero
print(house_density.groupby("search_area")["building_count_500m"].mean().round(0).sort_values(ascending=False))

search_area
New Road                   3324.0
Bhaktapur durbar square    2529.0
Baneshwor                  2408.0
Durbar Marg                2169.0
Boudha stupa               1965.0
Patan durbar square        1947.0
Pulchowk                   1528.0
Koteshwor                  1507.0
Kirtipur                   1147.0
Name: building_count_500m, dtype: float64


# final dataset

In [15]:
import pandas as pd

hd = pd.read_csv("../data/processed/sub_groups/house_density_features.csv")
dm = pd.read_csv("../data/processed/sub_groups/demand_features.csv")
ac = pd.read_csv("../data/processed/sub_groups/accessibility_features.csv")

id_cols = ["place_id", "restaurant_name", "latitude", "longitude", "search_area"]

# rebuild Accessibility score from raw columns (benefits + inverted cost)
benefit = ["bus_stop_count_500m", "parking_space_count_500m", "intersection_count_500m"]
cost    = ["dist_to_main_road_m"]
norm = pd.DataFrame(index=ac.index)
for c in benefit:
    mn, mx = ac[c].min(), ac[c].max()
    norm[c] = 0.0 if mx == mn else (ac[c] - mn) / (mx - mn)
for c in cost:
    mn, mx = ac[c].min(), ac[c].max()
    norm[c] = 1.0 if mx == mn else (mx - ac[c]) / (mx - mn)   # invert: near = good
ac["Accessibility"] = norm.mean(axis=1).round(4)

# merge, keeping ONLY the three group factors + identity
merged = (hd[id_cols + ["building_count_500m"]]
    .merge(dm[["place_id", "Demand"]], on="place_id")
    .merge(ac[["place_id", "Accessibility"]], on="place_id"))

assert len(merged) == 1472 and merged.place_id.nunique() == 1472, "row mismatch!"
merged.to_csv("../data/processed/features_merged.csv", index=False)
merged.head()

,place_id,restaurant_name,latitude,longitude,search_area,building_count_500m,Demand,Accessibility
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,JAR - Just Another Restaurant,27.699546,85.337687,Baneshwor,2695,0.2995,0.4165
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,27.692262,85.336472,Baneshwor,2489,0.4008,0.4903
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Pink Putali Restaurant & Bar,27.689064,85.334295,Baneshwor,2413,0.4458,0.5682
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Munch N More Restaurant and Bar,27.681549,85.341340,Baneshwor,3176,0.2319,0.5350
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27 Degree North Restaurant,27.688812,85.334080,Baneshwor,2437,0.4351,0.5848


# Final dataset including all factors

In [25]:
import pandas as pd

fm = pd.read_csv("../data/processed/features_merged.csv")
fd = pd.read_csv("../data/processed/final_dataset.csv")

comp = ["competitor_count_500m", "avg_restaurant_rating_500m", "avg_review_ratings_500m", "nearest_restaurant_m"]
c = fd[["place_id"] + comp].copy()

# isolated restaurants: 0 = no competitors nearby (meaningful absence, not missing)
c[comp] = c[comp].fillna(0)

benefit = ["nearest_restaurant_m"]
cost    = ["competitor_count_500m", "avg_restaurant_rating_500m", "avg_review_ratings_500m"]

norm = pd.DataFrame(index=c.index)
for col in benefit:
    mn, mx = c[col].min(), c[col].max()
    norm[col] = 0.0 if mx == mn else (c[col] - mn) / (mx - mn)
for col in cost:
    mn, mx = c[col].min(), c[col].max()
    norm[col] = 1.0 if mx == mn else (mx - c[col]) / (mx - mn)   # invert

c["Competition"] = norm.mean(axis=1).round(4)

# ---- drop any existing Competition so the merge doesn't collide ----
fm = fm.drop(columns=[x for x in ["Competition"] if x in fm.columns])

merged = fm.merge(c[["place_id", "Competition"]], on="place_id")
assert len(merged) == 1472 and merged.place_id.nunique() == 1472
merged.to_csv("../data/processed/features_merged.csv", index=False)
merged.head()

,place_id,restaurant_name,latitude,longitude,search_area,building_count_500m,Demand,Accessibility,Competition
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,JAR - Just Another Restaurant,27.699546,85.337687,Baneshwor,2695,0.2995,0.4165,0.4850
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,27.692262,85.336472,Baneshwor,2489,0.4008,0.4903,0.4213
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Pink Putali Restaurant & Bar,27.689064,85.334295,Baneshwor,2413,0.4458,0.5682,0.4101
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Munch N More Restaurant and Bar,27.681549,85.341340,Baneshwor,3176,0.2319,0.5350,0.4891
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27 Degree North Restaurant,27.688812,85.334080,Baneshwor,2437,0.4351,0.5848,0.4088


# building_count is raw so normalizing it

In [26]:
import pandas as pd

df = pd.read_csv("../data/processed/features_merged.csv")

# normalize building_count_500m -> Population score (benefit: more buildings = higher)
mn, mx = df["building_count_500m"].min(), df["building_count_500m"].max()
df["Population"] = 0.0 if mx == mn else ((df["building_count_500m"] - mn) / (mx - mn)).round(4)

df.to_csv("../data/processed/features_merged.csv", index=False)
df[["Demand", "Accessibility", "Competition", "Population"]].describe().round(3)

,Demand,Accessibility,Competition,Population
count,1472.000,1472.000,1472.000,1472.000
mean,0.269,0.419,0.433,0.387
std,0.158,0.117,0.103,0.227
min,0.000,0.000,0.236,0.000
25%,0.162,0.354,0.372,0.262
50%,0.258,0.415,0.448,0.366
75%,0.396,0.495,0.494,0.498
max,0.646,0.752,1.000,1.000
